In [9]:
import polars as pl

In [19]:
from hot_topic.utils import extract_show_and_episode

In [32]:
raw_df = pl.read_csv('redund.csv')
raw_df

valid_audio,percent_skipped,percent_replaced_text,percent_replaced_music,percent_appended,transcript
bool,f64,f64,f64,f64,str
true,0.084,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…"
true,0.103,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…"
true,0.006,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…"
true,0.049,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…"
true,0.034,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…"
…,…,…,…,…,…
true,0.001,0.0,0.0,0.0,"""out/whisper_segmented/108306/1…"
true,0.038,0.0,0.0,0.0,"""out/whisper_segmented/108306/1…"
true,0.061,0.0,0.0,0.004,"""out/whisper_segmented/108306/1…"


In [33]:
from operator import itemgetter
show_ep_pairs = list(map(extract_show_and_episode, raw_df['transcript']))


In [48]:
bad_threshold = 0.10
skip_threshold = 0.10
df = raw_df \
    .with_columns(
        show_id = pl.Series(map(itemgetter(0), show_ep_pairs)),
        episode_id = pl.Series(map(itemgetter(1), show_ep_pairs)),
        percent_replaced = pl.col('percent_replaced_text') + pl.col('percent_replaced_music')
    ) \
    .with_columns(
        replacement_err = pl.col('percent_replaced') >= bad_threshold,
        skip_err    = pl.col('percent_skipped')  >= skip_threshold,
        download_err = ~pl.col('valid_audio')
    ) \
    .with_columns(bad_transcript = pl.col('replacement_err') | pl.col('skip_err') | pl.col('download_err') )
df

valid_audio,percent_skipped,percent_replaced_text,percent_replaced_music,percent_appended,transcript,show_id,episode_id,percent_replaced,replacement_err,skip_err,download_err,bad_transcript
bool,f64,f64,f64,f64,str,i64,str,f64,bool,bool,bool,bool
true,0.084,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…",5647380,"""5647380_00031""",0.0,false,false,false,false
true,0.103,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…",5647380,"""5647380_00063""",0.0,false,true,false,true
true,0.006,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…",5647380,"""5647380_00001""",0.0,false,false,false,false
true,0.049,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…",5647380,"""5647380_00059""",0.0,false,false,false,false
true,0.034,0.0,0.0,0.0,"""out/whisper_segmented/5647380/…",5647380,"""5647380_00062""",0.0,false,false,false,false
…,…,…,…,…,…,…,…,…,…,…,…,…
true,0.001,0.0,0.0,0.0,"""out/whisper_segmented/108306/1…",108306,"""108306_00121""",0.0,false,false,false,false
true,0.038,0.0,0.0,0.0,"""out/whisper_segmented/108306/1…",108306,"""108306_00180""",0.0,false,false,false,false
true,0.061,0.0,0.0,0.004,"""out/whisper_segmented/108306/1…",108306,"""108306_00179""",0.0,false,false,false,false


In [ ]:
total_df = df.group_by('show_id').agg(pl.len())
bad_count_df = df.group_by('show_id').agg(
    pl.sum('replacement_err') / pl.len(),
    pl.sum('skip_err') / pl.len(),
    pl.sum('download_err') / pl.len(),
    pl.sum('any_err') / pl.len()
)

# joined_df = bad_count_df.join(total_df, on='show_id', how='left').with_columns(
#     replace_percentage         = pl.col('high_replace') / pl.col('len'),
#     skip_percentage            = pl.col('high_skip')    / pl.col('len'),
#     download_err_percentage    = pl.col('download_err') / pl.col('len'),
#     bad_percentage             = pl.col('bad_count')    / pl.col('len'),
# )
# joined_df
bad_count_df

show_id,high_replace,high_skip,download_err
i64,f64,f64,f64
620294,0.0,0.0,0.0
100698,0.0,0.0,0.0
3070848,0.0,0.0,0.0
5951054,0.05,0.043333,0.0
771880,0.009709,0.0,0.0
…,…,…,…
6067218,0.014286,0.0,0.0
1017776,0.186235,0.004049,0.0
4117081,0.0,0.0,0.0


In [43]:
joined_df.write_csv('bad_shows.csv')

In [17]:
import plotly.express as px

In [18]:
px.histogram(sub_df, x='percent_replaced')